<a href="https://colab.research.google.com/github/frandresst-wq/Proyecto-de-Ciencia-de-Datos/blob/main/S1_Notebook_Base_CDD_Soto_Franco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 1 — Obtención del dato
**Ciencia de Datos · Ingeniería de Ejecución en Informática — Advance UCSC**

| | |
|---|---|
| **Semana** | 1 |
| **Resultado de aprendizaje** | RA1 (foco) · RA3 (transversal) · RA5 (apertura) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Estudiante** | _(Franco Soto)_ |

---

## Antes de empezar

Sube al entorno los tres archivos que acompañan esta guía:

- `S1_Dataset_Guia_CDD.csv`
- `S1_Base_Ejemplo_CDD.sqlite`
- `S1_Respuesta_API_Ejemplo_CDD.json`

En Google Colab: panel izquierdo → icono de carpeta → *Subir*. En Jupyter local: déjalos en la misma carpeta que este cuaderno.

## Celda 0 · Pregunta de activación

Antes de escribir código, responde aquí para ti (no se entrega):

> **¿Qué dato de tu propio trabajo no usarías hoy para tomar una decisión importante, y por qué?**

_Nombre de Pacientes,Diagnosticos medicos. Por que vulnera la privacidad de las personas_

## 1 · Preparación del entorno

Un flujo reproducible parte por declarar qué versiones se usaron. Si alguien ejecuta este cuaderno dentro de un año y obtiene otro resultado, lo primero que va a mirar es esto.

In [1]:
import sys, platform
import pandas as pd
import numpy as np

RUTA = ''          # en Colab déjalo vacío; en local, la carpeta donde están los archivos
SEMILLA = 20261
np.random.seed(SEMILLA)

print('Python    :', sys.version.split()[0], '·', platform.system())
print('pandas    :', pd.__version__)
print('numpy     :', np.__version__)
print('semilla   :', SEMILLA)

Python    : 3.13.16 · Linux
pandas    : 2.2.3
numpy     : 2.1.3
semilla   : 20261


## 2 · Fuente 1 — Archivo plano (CSV)

El primer problema de un archivo plano nunca es el análisis: es **abrirlo bien**. Tres cosas pueden salir mal antes de ver un solo número: la codificación, el separador y los tipos de dato.

Ejecuta la celda siguiente y observa el resultado. Después responde: ¿cuál es el separador de este archivo?

In [3]:
# Lectura deliberadamente ingenua: mira lo que pasa
try:
    crudo = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv')
    print('Forma:', crudo.shape)
    display(crudo.head(3))
except Exception as e:
    print(type(e).__name__)
    print(e)

ParserError
Error tokenizing data. C error: Expected 1 fields in line 39, saw 2



**Lee el error, no lo saltes.** Dice algo como *«Expected 1 fields in line 39, saw 2»*.

Se traduce así: pandas asumió que el separador era la coma, no encontró ninguna en la primera línea y concluyó que el archivo tenía **una sola columna**. Todo iba bien hasta la línea 39, donde apareció una coma —un decimal escrito a la chilena, `12,4`— y esa fila pasó a tener dos campos. Contradicción, y se detiene.

Son dos defectos a la vez: el separador real es `;` y además hay decimales con coma. Resolvamos primero el separador.

In [ ]:
aire = pd.read_csv(
    RUTA + 'S1_Dataset_Guia_CDD.csv',
    sep=';',
    encoding='utf-8',
)
print('Forma:', aire.shape)
aire.head(40)

In [ ]:
# Tipos de dato con los que pandas interpretó cada columna
aire.dtypes

### ✏️ Actividad 2.1

Mira los tipos con atención. `pm25` y `pm10` quedaron numéricos, pero `temperatura_c` quedó como `object`, es decir, texto. Que una columna numérica quede como texto **siempre** significa que hay algo dentro que no es un número: aquí son los nueve decimales escritos con coma que provocaron el error anterior.

Y existe la trampa inversa: `pm10` es numérico y aun así está contaminado. Lo verás en la actividad 3.2.

**Escribe el código** que muestre los valores no numéricos de `temperatura_c`. Pista: `pd.to_numeric(..., errors='coerce')` convierte a `NaN` lo que no puede convertir.

In [9]:
# Tu código aquí
aire[pd.to_numeric(aire['temperatura_c'], errors='coerce').isna()]


,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado
37,02-20260603,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-06-03,69.6,115.0,"7,8",93.0,SI
209,07-20260615,7,Valdivia Centro,Valdivia,Los Ríos,2026-06-15,66.8,139.7,"8,3",92.9,SI
347,08-20260517,8,Coronel Norte,Coronel,Biobío,2026-05-17,45.3,76.6,"15,0",77.9,SI
429,03-20260514,3,Coyhanco,Chillán,Ñuble,14/05/2026,67.8,124.4,"10,5",89.3,SI
522,04-20260515,4,Talcahuano,Talcahuano,Biobío,2026-05-15,41.9,75.8,"8,8",78.5,SI
563,07-20260518,7,Valdivia Centro,Valdivia,Los Ríos,2026-05-18,62.1,106.9,"13,3",81.8,SI
603,05-20260408,5,Temuco Padre Las Casas,Temuco,Araucanía,2026-04-08,73.9,139.9,"16,4",75.6,SI
618,04-20260426,4,Talcahuano,Talcahuano,Biobío,2026-04-26,41.5,71.4,"10,1",56.7,SI
702,03-20260626,3,Coyhanco,Chillán,Ñuble,2026-06-26,65.3,118.6,"10,4",82.5,SI


## 3 · Perfilado inicial: ¿qué tengo entre manos?

Antes de analizar, hay que **conocer el dato**. Estas cuatro celdas son el mínimo que un analista ejecuta siempre, en este orden.

In [10]:
# a) Dimensiones y nombres de columna
print('Filas   :', aire.shape[0])
print('Columnas:', aire.shape[1])
print()
print(aire.columns.tolist())

Filas   : 734
Columnas: 11

['id_registro', 'estacion_id', 'estacion', 'comuna', 'region', 'fecha', 'pm25', 'pm10', 'temperatura_c', 'humedad_pct', 'validado']


In [11]:
# b) Valores faltantes por columna
faltantes = aire.isna().sum().to_frame('faltantes')
faltantes['porcentaje'] = (faltantes['faltantes'] / len(aire) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,porcentaje
pm25,35,4.77
estacion_id,0,0.00
id_registro,0,0.00
estacion,0,0.00
comuna,0,0.00
region,0,0.00
fecha,0,0.00
pm10,0,0.00
temperatura_c,0,0.00
humedad_pct,0,0.00


In [12]:
# c) Filas duplicadas completas
print('Duplicados exactos:', aire.duplicated().sum())
print('id_registro repetidos:', aire['id_registro'].duplicated().sum())

Duplicados exactos: 14
id_registro repetidos: 14


In [ ]:
# d) Cardinalidad de las columnas categóricas
for col in ['estacion', 'comuna', 'region', 'validado']:
    print(f'{col:15s} · {aire[col].nunique():3d} valores distintos')

### ✏️ Actividad 3.1

La columna `comuna` declara más valores distintos de los que deberían existir. **Ejecuta** `aire['comuna'].unique()` y explica en la celda de texto siguiente por qué ocurre.

Ojo: no lo corrijas todavía. Esta semana **detectamos**; en la semana 2 **corregimos** y dejamos registrada la decisión en la bitácora.

In [13]:
aire['comuna'].unique()

array(['Los Ángeles', 'Coronel', 'Osorno', 'Concepción', 'Temuco',
       'Talcahuano', 'Chillán', 'Valdivia', 'TEMUCO  ', 'VALDIVIA  ',
       'LOS ÁNGELES  ', 'TALCAHUANO  ', 'CHILLÁN  ', 'CONCEPCIÓN  ',
       'CORONEL  ', 'OSORNO  '], dtype=object)

_Tu explicación aquí:_



### ✏️ Actividad 3.2 — Valores centinela

Un **valor centinela** es un número real que en realidad significa «no hay dato»: `-999`, `0`, `9999`. Si no lo detectas, entra al promedio y lo arruina.

**Escribe el código** que cuente cuántas veces aparece `-999` en `pm10` y calcula el promedio de `pm10` con y sin esos valores. Anota la diferencia.

In [15]:
# Tu código aquí
# Contar cuantas veces aparece -999
cantidad = (aire['pm10'] ==  -999).sum()

#Promedio incluyendo -999
promedio_con = aire['pm10'].mean()

#Promedio excluyendo -999
promedio_sin = aire.loc[aire['pm10'] != -999, 'pm10'].mean()

#Calcular la diferencia
diferencia = promedio_sin - promedio_con

print("Cantidad de -999:", cantidad)
print("Promedio con -999:", round(promedio_con, 2))
print("Promedio sin -999:", round(promedio_sin, 2))
print("Diferencia:", round(diferencia,2))




Cantidad de -999: 18
Promedio con -999: 77.1
Promedio sin -999: 104.15
Diferencia: 27.05


## 4 · Fuente 2 — Base de datos relacional (SQL)

Buena parte de los datos de una organización no está en archivos, sino en una base de datos. El archivo `S1_Base_Ejemplo_CDD.sqlite` contiene tres tablas: `estacion`, `medicion` y `episodio`.

SQLite viene incluido en Python: no hay que instalar nada.

In [16]:
import sqlite3

con = sqlite3.connect(RUTA + 'S1_Base_Ejemplo_CDD.sqlite')

# ¿Qué tablas hay?
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,estacion
1,medicion
2,sqlite_sequence
3,episodio


In [17]:
# Estructura de una tabla
pd.read_sql_query('PRAGMA table_info(medicion)', con)

,cid,name,type,notnull,dflt_value,pk
0,0,medicion_id,INTEGER,0,None,1
1,1,estacion_id,INTEGER,1,None,0
2,2,fecha,TEXT,1,None,0
3,3,pm25,REAL,0,None,0
4,4,pm10,REAL,0,None,0
5,5,temperatura_c,REAL,0,None,0
6,6,humedad_pct,REAL,0,None,0


In [18]:
# Una consulta con JOIN y agregación: promedio de PM2.5 por comuna
consulta = '''
SELECT  e.comuna,e.region,COUNT(*) AS n_mediciones, ROUND(AVG(m.pm25), 1) AS pm25_promedio,ROUND(MAX(m.pm25), 1) AS pm25_maximo
FROM    medicion m
JOIN    estacion e ON e.estacion_id = m.estacion_id
GROUP BY e.comuna, e.region
ORDER BY pm25_promedio DESC
'''
pd.read_sql_query(consulta, con)

,comuna,region,n_mediciones,pm25_promedio,pm25_maximo
0,Temuco,Araucanía,90,74.2,102.1
1,Chillán,Ñuble,90,65.8,98.5
2,Los Ángeles,Biobío,90,62.2,87.5
3,Osorno,Los Lagos,90,60.3,82.5
4,Valdivia,Los Ríos,90,53.9,78.0
5,Coronel,Biobío,90,49.0,76.5
6,Concepción,Biobío,90,45.8,70.7
7,Talcahuano,Biobío,90,41.3,67.8


### ✏️ Actividad 4.1

**Escribe una consulta SQL** que responda: *¿cuántos episodios de cada categoría (`Alerta`, `Preemergencia`, `Emergencia`) registró cada comuna?*

Necesitas unir `episodio` con `estacion` y agrupar por dos columnas.

In [22]:
consulta_2 = '''
SELECT comuna,categoria,count(*) as cantidad
FROM estacion es
JOIN episodio ep ON es.estacion_id = ep.estacion_id
GROUP BY comuna,categoria
ORDER BY comuna,categoria;

'''
pd.read_sql_query(consulta_2,con)

,comuna,categoria,cantidad
0,Chillán,Alerta,5
1,Chillán,Preemergencia,1
2,Los Ángeles,Alerta,6
3,Osorno,Alerta,5
4,Temuco,Alerta,25
5,Temuco,Preemergencia,4


In [ ]:
con.close()

## 5 · Fuente 3 — Servicio web (API) y JSON anidado

Una API devuelve **JSON**, no una tabla. El JSON es jerárquico: dentro de un registro puede haber otro objeto, y dentro de ese, una lista. Convertirlo en una tabla rectangular es parte del trabajo.

Usamos aquí la respuesta guardada en archivo. En un caso real la obtendrías con `requests.get(url).json()`; el archivo existe para que el laboratorio funcione aunque el servicio esté caído.

In [23]:
import json

with open(RUTA + 'S1_Respuesta_API_Ejemplo_CDD.json', encoding='utf-8') as fh:
    respuesta = json.load(fh)

print('Claves de primer nivel:', list(respuesta.keys()))
print()
print('Metadatos:')
for k, v in respuesta['meta'].items():
    print(f'  {k}: {v}')

Claves de primer nivel: ['meta', 'data']

Metadatos:
  servicio: API de ejemplo · Red de monitoreo de calidad del aire (simulada para docencia)
  version: 1.2
  generado: 2026-09-21T10:00:00-03:00
  licencia: CC BY 4.0 (conjunto sintético de uso docente)
  paginacion: {'pagina': 1, 'por_pagina': 100, 'total': 100, 'siguiente': None}


In [24]:
# Un registro cualquiera, para entender la estructura
print(json.dumps(respuesta['data'][0], ensure_ascii=False, indent=2))

{
  "codigo_estacion": "EST-001",
  "ubicacion": {
    "comuna": "Concepción",
    "region": "Biobío",
    "coordenadas": {
      "lat": -36.827,
      "lon": -73.0498
    }
  },
  "fecha": "2026-04-01",
  "mediciones": [
    {
      "parametro": "pm25",
      "valor": 37.6,
      "unidad": "ug/m3"
    },
    {
      "parametro": "pm10",
      "valor": 58.8,
      "unidad": "ug/m3"
    },
    {
      "parametro": "temperatura",
      "valor": 13.1,
      "unidad": "C"
    }
  ],
  "calidad_dato": "preliminar"
}


In [25]:
# json_normalize aplana la jerarquía: fíjate en los nombres de columna que genera
api = pd.json_normalize(respuesta['data'])
api.head(3)

,codigo_estacion,fecha,mediciones,calidad_dato,ubicacion.comuna,ubicacion.region,ubicacion.coordenadas.lat,ubicacion.coordenadas.lon
0,EST-001,2026-04-01,"[{'parametro': 'pm25', 'valor': 37.6, 'unidad'...",preliminar,Concepción,Biobío,-36.827,-73.0498
1,EST-001,2026-04-02,"[{'parametro': 'pm25', 'valor': 25.6, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498
2,EST-001,2026-04-03,"[{'parametro': 'pm25', 'valor': 29.3, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498


In [26]:
# La lista 'mediciones' sigue anidada. record_path la despliega en filas:
api_largo = pd.json_normalize(
    respuesta['data'],
    record_path='mediciones',
    meta=['codigo_estacion', 'fecha', ['ubicacion', 'comuna']],
)
api_largo.head(6)

,parametro,valor,unidad,codigo_estacion,fecha,ubicacion.comuna
0,pm25,37.6,ug/m3,EST-001,2026-04-01,Concepción
1,pm10,58.8,ug/m3,EST-001,2026-04-01,Concepción
2,temperatura,13.1,C,EST-001,2026-04-01,Concepción
3,pm25,25.6,ug/m3,EST-001,2026-04-02,Concepción
4,pm10,63.5,ug/m3,EST-001,2026-04-02,Concepción
5,temperatura,15.6,C,EST-001,2026-04-02,Concepción


### ✏️ Actividad 5.1

El resultado anterior está en **formato largo**: una fila por parámetro medido. Para analizar conviene el **formato ancho**: una fila por estación-fecha y una columna por parámetro.

**Escribe el código** que haga esa transformación con `pivot_table`.

In [28]:
# Tu código aquí
api_ancho = api_largo.pivot_table(
    index=['codigo_estacion', 'fecha', 'ubicacion.comuna'],
    columns='parametro',
    values='valor'
).reset_index()

api_ancho.columns.name = None
api_ancho.head(6)


,codigo_estacion,fecha,ubicacion.comuna,pm10,pm25,temperatura
0,EST-001,2026-04-01,Concepción,58.8,37.6,13.1
1,EST-001,2026-04-02,Concepción,63.5,25.6,15.6
2,EST-001,2026-04-03,Concepción,75.4,29.3,13.3
3,EST-001,2026-04-04,Concepción,55.9,33.0,11.7
4,EST-001,2026-04-05,Concepción,79.1,39.1,12.4
5,EST-001,2026-04-06,Concepción,76.4,42.9,16.3


## 6 · Fuente 4 — Extracción web

Cuando el dato solo existe publicado en una página, hay que extraerlo. `pandas.read_html` resuelve el caso más común: una tabla HTML.

**Antes de extraer, tres preguntas que no son técnicas:** ¿los términos de uso del sitio lo permiten?, ¿estás generando una carga razonable?, ¿el dato incluye personas? Si alguna respuesta te incomoda, no extraigas.

In [32]:

import pandas as pd
import requests
from io import StringIO

URL = 'https://es.wikipedia.org/wiki/Anexo:Comunas_de_Chile'

try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    respuesta = requests.get(URL, headers=headers, timeout=20)
    respuesta.raise_for_status()
    tablas = pd.read_html(StringIO(respuesta.text))

    print('Tablas encontradas:', len(tablas))
    web = tablas[0]
    display(web.head())
except Exception as e:
    print('No se pudo extraer:', type(e).__name__, e)
    print('Continúa con el resto del laboratorio.')

Tablas encontradas: 2


,CUT (Código Único Territorial),Nombre,Unnamed: 2,Provincia,Región,Superficie (km²),Población 2024 (Censo),Densidad (hab./km²),IDH 2005,IDH 2005.1,Latitud,Longitud
0,15101,Arica,NaN,Arica,Arica y Parinacota,"4.799,4",241 653,51.6,0.866,Muy alto,"-18°27'18""","-70°17'24"""
1,15102,Camarones,NaN,Arica,Arica y Parinacota,3.927,861,0.31,0.791,Alto,"-19°1'1.2""","-69°52'1.2"""
2,15201,Putre,NaN,Parinacota,Arica y Parinacota,"5.902,5",1 547,0.43,0.817,Muy alto,"-18°12'0""","-69°34'58.8"""
3,15202,General Lagos,NaN,Parinacota,Arica y Parinacota,"2.244,4",508,0.36,0.773,Alto,"-17°39'10.8""","-69°38'6"""
4,1101,Iquique,NaN,Iquique,Tarapacá,"2.242,1",199 587,996,0.826,Muy alto,"-20°14'38.4""","-70°8'20.4"""


### ✏️ Actividad 6.1

Responde en la celda de texto siguiente: **¿qué diferencia hay, para tu proyecto, entre un dato descargado de un portal oficial y uno extraído de una página web?** Piensa en trazabilidad y en qué pasa si mañana la página cambia.

_Un dato extraido de una fuente oficial es un dato que ofrece mayor confiabilidad,trazabilidad y estabilidad ya que la informacion es entregada por una institucion confiable y suele incluir informacion de su origen en cambio un dato extraido de una pagina web puede ser modificado y/o eliminado en cualquier momento dificultando su uso, recuperacion o actualizacion automatica. Por eso para mi proyecto de ciencia de datos esta diferencia es importante porque necesito trabajar con datos verificable y reproducible. Si mañana cambia la pagina web podria fallar el proceso de extraccion de datos o variar la informacion obtenida. Por eso es muy importante registrar la fuente , la fecha de extraccion y conservar una copia de los datos originales, independientemente de su procedencia._



## 7 · Volumen: cuando el conjunto no cabe

No todo conjunto se lee entero. Cuando el archivo supera la memoria disponible hay tres salidas: **leer solo las columnas necesarias**, **leer por bloques** o **trabajar con una muestra**.

Una muestra bien construida responde muchas preguntas con una fracción del costo. Una muestra mal construida responde mal todas.

In [33]:
# Leer solo algunas columnas: menos memoria desde el inicio
liviano = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';',
                      usecols=['estacion', 'fecha', 'pm25'])
print('Memoria completo :', aire.memory_usage(deep=True).sum() / 1024, 'KB')
print('Memoria liviano  :', liviano.memory_usage(deep=True).sum() / 1024, 'KB')

Memoria completo : 335.0009765625 KB
Memoria liviano  : 94.8486328125 KB


In [34]:
# Lectura por bloques: se procesa de a trozos y nunca se carga todo
total, filas = 0, 0
for bloque in pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';', chunksize=100):
    valores = pd.to_numeric(bloque['pm25'], errors='coerce').dropna()
    total += valores.sum()
    filas += len(valores)
print(f'Promedio de pm25 calculado por bloques: {total / filas:.2f}')

Promedio de pm25 calculado por bloques: 62.04


In [35]:
# Muestreo aleatorio simple vs. estratificado por estación
simple = aire.sample(n=120, random_state=SEMILLA)
estratificado = (aire.groupby('estacion', group_keys=False)
                     .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))

print('Distribución en la muestra SIMPLE:')
print(simple['estacion'].value_counts().to_string())
print()
print('Distribución en la muestra ESTRATIFICADA:')
print(estratificado['estacion'].value_counts().to_string())

Distribución en la muestra SIMPLE:
estacion
Kingston                  18
Osorno Entre Lagos        18
Coyhanco                  17
Los Ángeles Oriente       17
Coronel Norte             16
Talcahuano                13
Temuco Padre Las Casas    13
Valdivia Centro            8

Distribución en la muestra ESTRATIFICADA:
estacion
Coronel Norte             15
Coyhanco                  15
Kingston                  15
Los Ángeles Oriente       15
Osorno Entre Lagos        15
Talcahuano                15
Temuco Padre Las Casas    15
Valdivia Centro           15


/tmp/ipykernel_5921/748870216.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))


### ✏️ Actividad 7.1

Mira las dos distribuciones anteriores. **¿En qué situación la muestra simple te llevaría a una conclusión equivocada?** Escríbelo abajo con un ejemplo concreto de tu propio conjunto de datos.

_El muestreo aleatorio simple selecciona registros sin garantizar una cantidad equilibrada por estación, mientras que el muestreo estratificado permite asegurar la participación de todas las estaciones con igual número de observaciones. Para este proyecto de calidad del aire, el muestreo estratificado es más conveniente si el objetivo es comparar las condiciones ambientales entre estaciones, porque evita que algunas tengan mayor representación que otras. Sin embargo, si se busca estimar un promedio global representativo de toda la población, se deben considerar las proporciones originales de cada estación._



## 8 · Tu conjunto de datos: la elección del proyecto

Abre `S1_Catalogo_Datasets_Sugeridos_CDD.xlsx`, revisa **al menos tres** alternativas y elige una. Registra tu elección en la hoja «Mi elección» del mismo archivo.

Luego descárgalo, cárgalo aquí abajo y ejecuta sobre él el mismo perfilado de la sección 3.

In [38]:
# mi_dataset = pd.read_csv(RUTA + 'NOMBRE_DE_TU_ARCHIVO.csv')
# print(mi_dataset.shape)
# mi_dataset.head()
mi_dataset = pd.read_excel(RUTA + 'S1_Catalogo_Datasets_Sugeridos_CDD.xlsx',header=4)
mi_dataset = mi_dataset[pd.to_numeric(mi_dataset['N°'], errors='coerce').notna()].copy() #elimina filas con N° no numeros
print(mi_dataset.shape)
mi_dataset.head(30)

(12, 10)


,N°,Nombre del conjunto de datos,Portal / fuente,Tema,Formato,Tamaño aprox.,¿Datos personales?,Dificultad,Pregunta analítica sugerida,Enlace (verificar antes del montaje)
0,1,Calidad del aire — estaciones de monitoreo (SI...,Ministerio del Medio Ambiente / datos.gob.cl,Medio ambiente,CSV,> 100.000 filas,No,Media,¿Qué estaciones concentran los episodios críti...,https://datos.gob.cl — buscar «calidad del aire»
1,2,Estadísticas de accidentes de tránsito,CONASET / datos.gob.cl,Transporte y seguridad,CSV / XLSX,50.000–200.000 filas,No (agregados),Media,¿Qué factores se asocian a los tramos horarios...,https://datos.gob.cl — buscar «accidentes de t...
2,3,Encuesta Nacional de Empleo (ENE),Instituto Nacional de Estadísticas,Empleo,CSV,> 300.000 filas,No (anonimizada),Alta,¿Cómo varía la tasa de ocupación por región y ...,https://www.ine.gob.cl — sección Estadísticas ...
3,4,Índices de precios al consumidor (IPC),Instituto Nacional de Estadísticas,Economía,CSV / XLSX,Serie mensual larga,No,Baja,¿Qué divisiones del IPC explican la variación ...,https://www.ine.gob.cl — sección IPC
4,5,Series económicas del Banco Central,Banco Central de Chile,Economía,CSV / API,Series mensuales,No,Baja,¿Qué relación muestran el tipo de cambio y el ...,https://si3.bcentral.cl
5,6,Matrícula y rendimiento escolar,Ministerio de Educación / datos.gob.cl,Educación,CSV,> 3.000.000 filas,Sí (requiere cuidado),Alta,¿Qué características del establecimiento se as...,https://datosabiertos.mineduc.cl
6,7,Online Retail — transacciones de comercio elec...,UCI Machine Learning Repository,Retail,XLSX / CSV,≈ 540.000 filas,Sí (ID de cliente),Media,¿Qué productos se compran juntos y qué segment...,https://archive.ics.uci.edu — buscar «Online R...
7,8,Bike Sharing — demanda horaria de bicicletas,UCI Machine Learning Repository,Transporte,CSV,≈ 17.000 filas,No,Baja,¿Cómo influyen el clima y el día de la semana ...,https://archive.ics.uci.edu — buscar «Bike Sha...
8,9,Wine Quality — características fisicoquímicas,UCI Machine Learning Repository,Industria alimentaria,CSV,≈ 6.500 filas,No,Baja,¿Qué variables fisicoquímicas se asocian a la ...,https://archive.ics.uci.edu — buscar «Wine Qua...
9,10,Compras públicas — órdenes de compra,ChileCompra / datos.gob.cl,Sector público,CSV,> 1.000.000 filas,No (organismos),Alta,¿Qué organismos concentran el gasto y cómo se ...,https://datos.gob.cl — buscar «ChileCompra»


In [40]:
# Perfilado de tu conjunto: repite aqui las cuatro celdas de la sección 3
#Tus respuestas desde aqui:
# a) Dimensiones y nombres de columna
print('filas :' , mi_dataset.shape[0])
print('Columnas:' , mi_dataset.shape[1])
print()
print(mi_dataset.columns.tolist())


filas : 12
Columnas: 10

['N°', 'Nombre del conjunto de datos', 'Portal / fuente', 'Tema', 'Formato', 'Tamaño aprox.', '¿Datos personales?', 'Dificultad', 'Pregunta analítica sugerida', 'Enlace (verificar antes del montaje)']


In [43]:
# b) Valores faltantes por columna
faltantes = mi_dataset.isna().sum().to_frame('faltantes')
faltantes['porcentaje'] = (faltantes['faltantes'] / len(mi_dataset) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,porcentaje
N°,0,0.0
Nombre del conjunto de datos,0,0.0
Portal / fuente,0,0.0
Tema,0,0.0
Formato,0,0.0
Tamaño aprox.,0,0.0
¿Datos personales?,0,0.0
Dificultad,0,0.0
Pregunta analítica sugerida,0,0.0
Enlace (verificar antes del montaje),0,0.0


### ✏️ Actividad 8.1 — Pregunta analítica y unidad de análisis

Completa estas dos líneas. Son la base de todo el curso y las vas a necesitar en la Entrega 1.

- **Mi pregunta analítica es:** _…_
- **Una fila de mi conjunto representa:** _…_

Criterio para saber si la pregunta está bien formulada: si al leerla puedes nombrar qué columnas necesitas para responderla, está bien. Si no, es demasiado vaga.

## 9 · Riesgo ético y legal

Antes de usar tu conjunto, completa la ficha `S1_Ficha_Riesgo_Etico_CDD.docx`. No es un trámite: si tu conjunto contiene datos de personas, lo que decidas ahí condiciona todo lo que puedes hacer en las semanas siguientes.

Esa ficha es el insumo de tu intervención en el **foro de debate** de esta semana y se cierra en la Sección C de la Entrega Final.

## 10 · Repositorio y control de versiones

Tu trabajo tiene que poder reconstruirse. Sube este cuaderno a un repositorio y deja el enlace aquí.

Desde Colab: *Archivo → Guardar una copia en GitHub*. Desde local:

In [ ]:
# Ejecuta esto en una terminal, no en el cuaderno:
# git init
# git add S1_Notebook_Base_CDD.ipynb
# git commit -m 'Laboratorio 1: obtención del dato'
# git remote add origin https://github.com/USUARIO/proyecto-ciencia-datos.git
# git push -u origin main


**Enlace a mi repositorio:** _…_

---

## Cierre del laboratorio

Al terminar deberías tener:

- [ ] El entorno funcionando y las versiones declaradas
- [ ] Datos leídos desde las cuatro fuentes
- [ ] El perfilado inicial del conjunto guía, con sus defectos detectados (no corregidos)
- [ ] Tu conjunto de datos elegido y registrado en el catálogo
- [ ] Tu pregunta analítica y tu unidad de análisis escritas
- [ ] La ficha de riesgo ético-legal completada
- [ ] El repositorio creado y este cuaderno versionado

Lo que quede pendiente, llévalo a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 1_